# VEDJE in a few minutes

VEDJE reranks the candidates of a first-stage video retriever with a 33M-parameter joint encoder. Each video is encoded once into a small cache of a few tokens per sampled frame, so indexed videos need no encoding at query time.

[Project page](https://gabrieleserussi.github.io/vedje/) · [Code](https://github.com/GabrieleSerussi/vedje)

This tour indexes one sample clip with the released code and measures what its cache stores. It shows the feature-change targets of training and how the reranker reads a query, then plots the paper's results. It runs in about two minutes on a free CPU runtime.

In [ ]:
#@title Setup: the repository, its dependencies and the sample clip
cache_size = "64 tokens (48 KiB)"  #@param ["64 tokens (48 KiB)", "16 tokens (12 KiB)"]
my_video = ""  #@param {type:"string"}
#@markdown Leave `my_video` empty to use the sample clip, or give the path or URL of a video file of your own.

import importlib.metadata, importlib.util, os, subprocess, sys, tempfile, time, urllib.request, warnings
start = time.time()
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore")
REPO = os.environ.get("VEDJE_REPO", "/content/vedje")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/GabrieleSerussi/vedje.git", REPO], check=True)

def version_of(name):
    try:
        return tuple(int(part) for part in importlib.metadata.version(name).split(".")[:2])
    except Exception:
        return (0, 0)

needed = []
if version_of("transformers") < (5, 13):
    needed.append("transformers>=5.13")  # the first release with the VideoPrism models
if importlib.util.find_spec("ruamel") is None:
    needed.append("ruamel.yaml")
if my_video and importlib.util.find_spec("av") is None:
    needed.append("av")
if needed:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)

sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
import torch
from vedje import cache, config, delta, features, model, paper

M = 4 if cache_size.startswith("64") else 1  # tokens per sampled frame
source = my_video or os.path.join(REPO, "colab", "sample")
if source.startswith(("http://", "https://")):
    local = os.path.join(tempfile.gettempdir(), "my_video" + (os.path.splitext(source.split("?")[0])[1] or ".mp4"))
    request = urllib.request.Request(source, headers={"User-Agent": "vedje-colab-tour/1.0 (https://github.com/GabrieleSerussi/vedje)"})
    with urllib.request.urlopen(request) as response, open(local, "wb") as out:
        out.write(response.read())
    source = local
frames = features.load_frames(source)  # 16 frames, 3 x 288 x 288, values in [0, 1]
print(f"Ready in {time.time() - start:.0f} s: {frames.shape[0]} frames from {'your video' if my_video else 'the sample clip'}, "
      f"a cache of {16 * M} tokens.")

## A frozen encoder reads each video once

The frozen VideoPrism-B encoder turns each sampled frame into a grid of patch features $X_t$. Shown as colours, their first three principal components separate the peacock from the grass and the floor. This runs once per video, when the video is indexed, and never at query time.

In [ ]:
#@title Index the clip with the frozen encoder
import matplotlib.pyplot as plt
BLUE, GRAY = "#2548F6", "#6A6A6A"

backbone = features.load_backbone()
start = time.time()
patches, v_global = features.extract_patch_features(backbone, frames)  # (1, 16 * 256, 768)
X = patches[0].view(16, 256, -1)
print(f"Patch features X: {X.shape[0]} frames x {X.shape[1]} patches x {X.shape[2]} channels, "
      f"computed in {time.time() - start:.1f} s on {'a GPU' if patches.is_cuda else 'the CPU'}.")

# the first three principal components of all 16 x 256 patch features, shown as colours
flat = patches[0].float() - patches[0].float().mean(0, keepdim=True)
_, _, V = torch.pca_lowrank(flat, q=6, center=False)
pc = flat @ V[:, :3]
lo, hi = pc.quantile(0.02, dim=0), pc.quantile(0.98, dim=0)
colours = ((pc - lo) / (hi - lo)).clamp(0, 1).view(16, 16, 16, 3).numpy()

shown = range(0, 16, 2)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax_frame, ax_pc, t in zip(axes[0], axes[1], shown):
    ax_frame.imshow(frames[t].clamp(0, 1).permute(1, 2, 0).numpy())
    ax_frame.set_title(f"Frame {t + 1}", fontsize=9)
    ax_pc.imshow(colours[t], interpolation="nearest")
    ax_frame.axis("off"); ax_pc.axis("off")
axes[0, 0].text(-30, 144, "Frames", rotation=90, va="center", ha="center", fontsize=9)
axes[1, 0].text(-1.9, 7.5, "Patch features", rotation=90, va="center", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## Each frame becomes a few tokens, kept in temporal order

The compressor reads the patch features of one frame at a time and writes a few learned tokens $Z_t$. The cache $Z(v) = [Z_1; \dots; Z_{16}]$ stacks them frame by frame. The release has no trained weights, so the compressor below is freshly initialised. Its output has the shape, the order and the size of a trained cache.

In [ ]:
#@title Write the cache and measure what each video stores
torch.manual_seed(0)
compressor = cache.FramewiseCrossAttentionCompressor(num_queries_per_frame=M).eval()
with torch.inference_mode():
    Z = compressor(patches)  # Z(v): (1, 16 * M, 384)
stored = Z.to(torch.bfloat16)  # the cache is stored at two bytes per element
nbytes = stored.numel() * stored.element_size()
assert nbytes == cache.cache_bytes(16, M, 384)
full = cache.cache_bytes(16, 257, 768)  # Table 9: 256 patch vectors and one frame vector per frame
print(f"Z(v): {Z.shape[1]} tokens of dimension {Z.shape[2]} ({M} per frame), {nbytes:,} bytes = {nbytes / 1024:g} KiB per video.")
print(f"The frame-and-patch features of the same video take {full:,} bytes = {full / 2**20:.2f} MiB, "
      f"{full / nbytes:.1f} times more.")

In [ ]:
#@title The cache, frame by frame
import io
from IPython.display import Image as ShowImage, display
from PIL import Image

steps = []
for t in range(16):
    fig = plt.figure(figsize=(9, 2.9), dpi=80)
    ax_f = fig.add_axes([0.01, 0.12, 0.25, 0.76]); ax_p = fig.add_axes([0.28, 0.12, 0.25, 0.76]); ax_c = fig.add_axes([0.57, 0.12, 0.42, 0.76])
    ax_f.imshow(frames[t].clamp(0, 1).permute(1, 2, 0).numpy()); ax_f.set_title(f"Frame {t + 1} of 16", fontsize=10)
    ax_p.imshow(colours[t], interpolation="nearest"); ax_p.set_title("Its patch features", fontsize=10)
    for i in range(16):  # one group of M tokens per frame, in temporal order
        for j in range(M):
            ax_c.add_patch(plt.Rectangle((i * 1.0 + 0.1, j * (0.8 / M) + 0.1), 0.8, 0.8 / M - 0.06,
                                         color=BLUE if i <= t else "#E3E3E3", alpha=1.0 if i == t else 0.75))
        ax_c.text(i + 0.5, -0.25, f"Z{i + 1}", ha="center", fontsize=7, color="#333333" if i <= t else "#9A9A9A")
    ax_c.set_xlim(0, 16); ax_c.set_ylim(-0.6, 1.25); ax_c.axis("off")
    ax_c.set_title(f"Ordered cache, stored per video: {cache.cache_bytes(t + 1, M, 384) / 1024:g} KiB", fontsize=10)
    for ax in (ax_f, ax_p):
        ax.axis("off")
    buf = io.BytesIO(); fig.savefig(buf, format="png"); plt.close(fig)
    steps.append(Image.open(buf).convert("P", palette=Image.ADAPTIVE))
gif = os.path.join(tempfile.gettempdir(), "vedje_cache.gif")
steps[0].save(gif, save_all=True, append_images=steps[1:] + [steps[-1]] * 3, duration=450, loop=0)
display(ShowImage(filename=gif))

## Feature-change targets compare each frame with a later one

During training only, a head reads the tokens $Z_t$ of one frame and predicts how the frozen patch features change $h$ sampled frames later, $\Delta_{t,h} = X_{t+h} - X_t$. The matrix shows how similar the patch features of two frames are; the outlined cells are the pairs that this target uses. The head is discarded after training.

In [ ]:
#@title Compute the targets with the repository's function
targets = delta.delta_targets(patches, num_frames=16, patches_per_frame=256, horizons=(3,))
pairs = delta.valid_pairs(16, (3,))

unit = torch.nn.functional.normalize(X.float(), dim=-1)
similarity = torch.einsum("ipd,jpd->ij", unit, unit).numpy() / X.shape[1]  # mean cosine similarity per patch position
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(similarity, cmap="Blues", vmin=similarity.min(), vmax=1)
for t, h in pairs:
    ax.add_patch(plt.Rectangle((t + h - 0.5, t - 0.5), 1, 1, fill=False, edgecolor="#E8A400", linewidth=1.6))
ticks = [0, 3, 7, 11, 15]
ax.set_xticks(ticks, [t + 1 for t in ticks]); ax.set_yticks(ticks, [t + 1 for t in ticks])
ax.set_xlabel("Frame"); ax.set_ylabel("Frame")
ax.set_title("Similarity of the patch features between frames", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Mean cosine similarity")
plt.tight_layout()
plt.show()

## The joint encoder reads the query next to the cache

At query time, the joint encoder reads the tokens of the query and the cache of one candidate as a single sequence. The first-stage score $\rho(q, v)$ is embedded and added to its output $c(q, v)$ before the score head, which gives $s_\theta(q, v)$. The candidates are sorted by this score.

In [ ]:
#@title Build the reranker of the paper's MSR-VTT configuration
cfg = config.load_config(os.path.join(REPO, "configs", "vedje_vp_msrvtt.yaml"))
cfg["num_queries_per_frame"] = M
import contextlib
with contextlib.redirect_stdout(io.StringIO()):  # build_model prints its configuration
    net = model.build_model(cfg, training=True).eval()
counts = model.parameter_counts(net)
parts = [("Joint encoder (MiniLM-L12-H384)", "joint_encoder", "query time"),
         ("First-stage score embedding e_rho", "prior", "query time"),
         ("Score head h_psi", "score_head", "query time"),
         ("Compressor C_eta", "compressor", "indexing"),
         ("Future-delta head F_omega", "future_predictor", "training only")]
for name, key, when in parts:
    print(f"{name:34s} {counts[key] / 1e6:7.2f}M parameters   {when}")

query = "a peacock walks along a wooden ledge"
pair = net._tokenize_text_vision_pair([query], Z)  # the sequence the joint encoder reads
ids = pair["input_ids"][0].tolist()
n_cache = sum(1 for i in ids if i == net.tokenizer.image_token_id)
text = net.tokenizer.convert_ids_to_tokens([i for i in ids if i != net.tokenizer.image_token_id])
print(f"\nQuery: {query!r}")
print(f"Joint-encoder input: {' '.join(text[:-1])} + {n_cache} cache tokens + {text[-1]}, {len(ids)} positions in all.")

## The paper's results at a glance

The left plot places VEDJE among published systems by accuracy and online parameters (Figure 1). The right plot shows the R@1 gain over each matched first stage on MSR-VTT (Table 1b).

In [ ]:
#@title Accuracy, parameters and gains
results = paper.load_results()
points = [r for r in results["figure_1_table_22"]["rows"] if r["in_figure_1"]]
order = ["VideoCLIP-XL (FT)", "VideoCLIP-XL (ZS)", "VideoPrism", "PE-Core-B"]  # the page's order
matched = sorted(results["table_1b"]["rows"], key=lambda r: order.index(r["first_stage"]))

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
for r in points:
    ours = r["figure_label"] == "VEDJE"
    a.scatter(r["online_parameters_millions"], r["t2v_r1"], s=110 if ours else 50, color=BLUE if ours else GRAY, zorder=3)
    a.annotate(r["figure_label"].replace(" (reprod.)", ""), (r["online_parameters_millions"], r["t2v_r1"]), xytext=(7, 4), textcoords="offset points",
               fontsize=9, color=BLUE if ours else "#333333", fontweight="bold" if ours else "normal",
               ha="right" if r["online_parameters_millions"] > 1000 else "left")
a.set_xscale("log")
a.set_xlim(45, 14000)
a.set_xticks([63, 157, 300, 1000, 3000, 7600], ["63M", "157M", "300M", "1B", "3B", "7.6B"])
a.minorticks_off()
a.set_xlabel("Online parameters (log scale)")
a.set_ylabel("MSR-VTT text-to-video R@1 (%)")
a.set_title("Accuracy against online parameters (Figure 1)", fontsize=11)
a.grid(alpha=0.3)

label = {"VideoCLIP-XL (ZS)": "VideoCLIP-XL, zero-shot", "VideoCLIP-XL (FT)": "VideoCLIP-XL, fine-tuned"}
names = [label.get(r["first_stage"], r["first_stage"]) for r in matched]
y = range(len(matched))
b.barh([i - 0.18 for i in y], [r["t2v"]["gain"] for r in matched], height=0.34, color=BLUE, label="Text to video")
b.barh([i + 0.18 for i in y], [r["v2t"]["gain"] for r in matched], height=0.34, color=BLUE, alpha=0.45, label="Video to text")
for i, r in enumerate(matched):
    b.text(r["t2v"]["gain"] + 0.1, i - 0.18, f"+{r['t2v']['gain']}", va="center", fontsize=9)
    b.text(r["v2t"]["gain"] + 0.1, i + 0.18, f"+{r['v2t']['gain']}", va="center", fontsize=9)
b.set_yticks(list(y), names)
b.invert_yaxis()
b.set_xlim(0, 8.5)
b.set_xlabel("R@1 gain over the matched first stage, points")
b.set_title("Gains on MSR-VTT (Table 1b)", fontsize=11)
b.legend(loc="lower right", fontsize=9, frameon=False)
for ax in (a, b):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

## Run VEDJE on your own videos

Install the repository with `pip install -e .`, train VEDJE on a GPU, then index your own videos and search them with text. The [README](https://github.com/GabrieleSerussi/vedje#your-data) shows the annotation format, and `configs/` holds the other datasets.

```bash
vedje train configs/vedje_vp_msrvtt.yaml
vedje index output/vedje_vp_msrvtt/checkpoint_03.pth my_videos/ --out my_index.pt
vedje search my_index.pt "a person walks across a field"
```

To repeat the first steps of this tour on your own clip, set `my_video` in the setup cell and run all cells again.

## Cite

```bibtex
@misc{wagner2026vedje,
  title={VEDJE: Video-Efficient Discriminative Joint Encoder for Scalable Video-Text Retrieval},
  author={Shahaf Wagner and Gabriele Serussi and Dan Ben Ami and Tomer Galanti and Chaim Baskin},
  year={2026},
  note={Preprint},
}
```

The sample clip is "Peacock walking and eating" by Mx. Granger on Wikimedia Commons, released under CC0 1.0.